In [182]:
#Starter-script for Prophet-konkurranse
#--------------------------------------
#
#- Leser train.csv
#- Leser test_features.csv
#- Trener Prophet-modell
#- Lager submission.csv

#Du kan forbedre:
#- features
#- skalering
#- lag-features
#- hyperparametre
#- log-transform
#- osv.

In [183]:
import pandas as pd
import numpy as np
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.preprocessing import StandardScaler

In [184]:
lagnsavn="jmn5"

In [185]:
# --------------------------------------------------
# 1. LES DATA
# --------------------------------------------------
print("Leser data ...")
train = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/bysykkel_train.csv", parse_dates=["ds"])
test = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/test_compete.csv", parse_dates=["ds"])
train = train.sort_values("ds")
test = test.sort_values("ds")
print("done!")

Leser data ...
done!


In [186]:
weather_cols = [
    "air_temperature",
    "wind_speed",
    "precipitation_amount"
]

train[weather_cols] = (
    train[weather_cols]
    .ffill()
    .bfill()
)

In [187]:
for col in weather_cols:
    train[f"{col}_lag1"] = train[col].shift(-1)
    train[f"{col}_lag2"] = train[col].shift(-2)
    test[f"{col}_lag1"] = test[col].shift(-1)
    test[f"{col}_lag2"] = test[col].shift(-2)

In [188]:
weather_cols = [
       'air_temperature', 'wind_speed',
       'precipitation_amount', 'air_temperature_lag1', 'air_temperature_lag2',
       'wind_speed_lag1', 'wind_speed_lag2', 'precipitation_amount_lag1',
       'precipitation_amount_lag2'
]

In [189]:
train[weather_cols] = (
    train[weather_cols]
    .ffill()
    .bfill()
)

In [190]:
test[weather_cols] = (
    test[weather_cols]
    .ffill()
    .bfill()
)

In [191]:
regressors = [
    "is_weekend", 
    "air_temperature",
    "precipitation_amount",
    "precipitation_amount_lag1",
    "precipitation_amount_lag2",
    'air_temperature_lag1', 'air_temperature_lag2',
    "month", 'wind_speed', 'wind_speed_lag1', 'wind_speed_lag2',
    "weekday", 'hour'
]

In [192]:
# --------------------------------------------------
# 4. DEFINER PROPHET
# --------------------------------------------------

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    seasonality_mode="multiplicative"
)

m.add_country_holidays(country_name='NO')

for r in regressors:
    m.add_regressor(r)

In [193]:
print("Trener modell ...")
m.fit(train[["ds", "y"] + regressors])

Trener modell ...


18:36:31 - cmdstanpy - INFO - Chain [1] start processing
18:37:20 - cmdstanpy - INFO - Chain [1] done processing


In [194]:
df_cv = cross_validation(
    m,
    initial='730 days',   # første treningsvindu
    period='90 days',     # hvor ofte vi flytter cutoff
    horizon='150 days'     # hvor langt frem vi tester
)

cv_metrics = performance_metrics(df_cv)
print(cv_metrics[["horizon", "mae", "rmse", "mape"]].head())

print("\nGjennomsnittlige score:")
print(f"MAE:  {cv_metrics['mae'].mean():.3f}")
print(f"RMSE: {cv_metrics['rmse'].mean():.3f}")
print(f"MAPE: {cv_metrics['mape'].mean():.3f}")

  0%|          | 0/19 [00:00<?, ?it/s]

18:37:20 - cmdstanpy - INFO - Chain [1] start processing
18:37:24 - cmdstanpy - INFO - Chain [1] done processing
18:37:24 - cmdstanpy - INFO - Chain [1] start processing
18:37:26 - cmdstanpy - INFO - Chain [1] done processing
18:37:26 - cmdstanpy - INFO - Chain [1] start processing
18:37:31 - cmdstanpy - INFO - Chain [1] done processing
18:37:31 - cmdstanpy - INFO - Chain [1] start processing
18:37:36 - cmdstanpy - INFO - Chain [1] done processing
18:37:37 - cmdstanpy - INFO - Chain [1] start processing
18:37:43 - cmdstanpy - INFO - Chain [1] done processing
18:37:43 - cmdstanpy - INFO - Chain [1] start processing
18:37:50 - cmdstanpy - INFO - Chain [1] done processing
18:37:51 - cmdstanpy - INFO - Chain [1] start processing
18:37:59 - cmdstanpy - INFO - Chain [1] done processing
18:37:59 - cmdstanpy - INFO - Chain [1] start processing
18:38:07 - cmdstanpy - INFO - Chain [1] done processing
18:38:07 - cmdstanpy - INFO - Chain [1] start processing
18:38:20 - cmdstanpy - INFO - Chain [1]

           horizon        mae       rmse      mape
0 17 days 15:00:00  35.340613  48.142961  2.054180
1 17 days 18:00:00  35.678260  48.588731  2.013095
2 17 days 21:00:00  35.832358  48.774367  2.027083
3 18 days 00:00:00  35.718999  48.675839  2.018860
4 18 days 06:00:00  35.673596  48.616274  2.118923

Gjennomsnittlige score:
MAE:  56.736
RMSE: 83.892
MAPE: 2.408


In [195]:
# --------------------------------------------------
# 6. PREDIKSJON PÅ TEST
# --------------------------------------------------
future = test[["ds"] + regressors].copy()
forecast = m.predict(future)

In [196]:
submission = pd.DataFrame({
    "ds": test["ds"],
    "yhat": forecast["yhat"]
})

# Unngå negative prediksjoner
submission["yhat"] = submission["yhat"].clip(lower=0)


In [197]:
#lagre innlevering
submission.to_csv(f"submission_{lagnsavn}.csv", index=False, sep=',')
print("✅ Ferdig!")
print("Submission lagret")

✅ Ferdig!
Submission lagret
